In [20]:
import os
from pathlib import Path

import pandas as pd

In [21]:
def load_all_data(data_dir: str | Path) -> dict:
    data_dir = Path(data_dir)

    file_aliases = {
        'customers': ['customers.csv'],
        'geography': ['geography.csv'],
        'inventory': ['inventory.csv'],
        'orderitems': ['orderitems.csv', 'order_items.csv'],
        'orders': ['orders.csv'],
        'payments': ['payments.csv'],
        'products': ['products.csv'],
        'promotions': ['promotions.csv'],
        'returns': ['returns.csv'],
        'reviews': ['reviews.csv'],
        'sales': ['sales.csv'],
        'samplesubmission': ['samplesubmission.csv', 'sample_submission.csv'],
        'shipments': ['shipments.csv'],
        'webtraffic': ['webtraffic.csv', 'web_traffic.csv'],
    }

    column_rename_maps = {
        'customers': {
            'customer_id': 'customerid',
            'signup_date': 'signupdate',
            'age_group': 'agegroup',
            'acquisition_channel': 'acquisitionchannel',
        },
        'orders': {
            'order_id': 'orderid',
            'order_date': 'orderdate',
            'customer_id': 'customerid',
            'order_status': 'orderstatus',
            'payment_method': 'paymentmethod',
            'device_type': 'devicetype',
            'order_source': 'ordersource',
        },
        'orderitems': {
            'order_id': 'orderid',
            'product_id': 'productid',
            'unit_price': 'unitprice',
            'discount_amount': 'discountamount',
            'promo_id': 'promoid',
            'promo_id_2': 'promoid2',
        },
        'payments': {
            'order_id': 'orderid',
            'payment_method': 'paymentmethod',
            'payment_value': 'paymentvalue',
        },
        'products': {
            'product_id': 'productid',
            'product_name': 'productname',
        },
        'promotions': {
            'promo_id': 'promoid',
            'promo_name': 'promoname',
            'promo_type': 'promotype',
            'discount_value': 'discountvalue',
            'start_date': 'startdate',
            'end_date': 'enddate',
            'applicable_category': 'applicablecategory',
            'promo_channel': 'promochannel',
            'stackable_flag': 'stackableflag',
            'min_order_value': 'minordervalue',
        },
        'returns': {
            'return_id': 'returnid',
            'order_id': 'orderid',
            'product_id': 'productid',
            'return_date': 'returndate',
            'return_reason': 'returnreason',
            'return_quantity': 'returnquantity',
            'refund_amount': 'refundamount',
        },
        'reviews': {
            'review_id': 'reviewid',
            'order_id': 'orderid',
            'product_id': 'productid',
            'customer_id': 'customerid',
            'review_date': 'reviewdate',
            'review_title': 'reviewtitle',
        },
        'inventory': {
            'snapshot_date': 'snapshotdate',
            'product_id': 'productid',
            'stock_on_hand': 'stockonhand',
            'units_received': 'unitsreceived',
            'units_sold': 'unitssold',
            'stockout_days': 'stockoutdays',
            'days_of_supply': 'daysofsupply',
            'fill_rate': 'fillrate',
            'stockout_flag': 'stockoutflag',
            'overstock_flag': 'overstockflag',
            'reorder_flag': 'reorderflag',
            'sell_through_rate': 'sellthroughrate',
            'product_name': 'productname',
        },
        'shipments': {
            'order_id': 'orderid',
            'ship_date': 'shipdate',
            'delivery_date': 'deliverydate',
            'shipping_fee': 'shippingfee',
        },
        'webtraffic': {
            'unique_visitors': 'uniquevisitors',
            'page_views': 'pageviews',
            'bounce_rate': 'bouncerate',
            'avg_session_duration_sec': 'avgsessiondurationsec',
            'traffic_source': 'trafficsource',
        },
    }

    dataframes = {}
    for table_name, candidates in file_aliases.items():
        file_path = next((data_dir / name for name in candidates if (data_dir / name).exists()), None)

        if file_path is None:
            print(f"Warning: File for table '{table_name}' not found. Checked: {candidates}")
            continue

        df = pd.read_csv(file_path, low_memory=False)
        df = df.rename(columns=column_rename_maps.get(table_name, {}))
        dataframes[table_name] = df
        print(f"Loaded {table_name} from {file_path.name}: {df.shape}")

    return dataframes

In [22]:
def format_datetime(dataframes: dict) -> dict:
    # Ánh xạ tên bảng và các cột mang ý nghĩa thời gian theo format của đề
    date_mapping = {
        'customers': ['signupdate'],
        'promotions': ['startdate', 'enddate'],
        'orders': ['orderdate'],
        'shipments': ['shipdate', 'deliverydate'],
        'returns': ['returndate'],
        'reviews': ['reviewdate'],
        'sales': ['Date'],
        'inventory': ['snapshotdate'],
        'webtraffic': ['date'],
    }

    for table, cols in date_mapping.items():
        if table not in dataframes:
            continue

        for col in cols:
            if col in dataframes[table].columns:
                # Ép kiểu datetime, coerce errors thành NaT nếu có lỗi parse
                dataframes[table][col] = pd.to_datetime(dataframes[table][col], errors='coerce')
            else:
                print(f"Warning: Column '{col}' not found in table '{table}'.")
    return dataframes

In [23]:
def optimize_datatypes(dataframes: dict) -> dict:
    categorical_cols = {
        'products': ['category', 'segment', 'size', 'color'],
        'orders': ['orderstatus', 'paymentmethod', 'devicetype', 'ordersource'],
    }

    for table, cols in categorical_cols.items():
        if table not in dataframes:
            continue
        for col in cols:
            if col in dataframes[table].columns:
                dataframes[table][col] = dataframes[table][col].astype('category')
            else:
                print(f"Warning: Column '{col}' not found in table '{table}'.")
    return dataframes

In [24]:
def handle_missing_values(dataframes: dict) -> dict:
    # Xử lý bảng customers
    if 'customers' in dataframes:
        cols_to_fill = ['gender', 'agegroup', 'acquisitionchannel']
        existing_cols = [col for col in cols_to_fill if col in dataframes['customers'].columns]
        if existing_cols:
            dataframes['customers'][existing_cols] = dataframes['customers'][existing_cols].fillna('Unknown')

    # Xử lý bảng orderitems
    if 'orderitems' in dataframes:
        for col in ['promoid', 'promoid2']:
            if col in dataframes['orderitems'].columns:
                dataframes['orderitems'][col] = dataframes['orderitems'][col].fillna('No_Promo')

    # Null nghĩa là áp dụng cho tất cả category
    if 'promotions' in dataframes and 'applicablecategory' in dataframes['promotions'].columns:
        dataframes['promotions']['applicablecategory'] = dataframes['promotions']['applicablecategory'].fillna('All')

    return dataframes

In [25]:
def preprocess_data(data_dir: str):
    df_dict = load_all_data(data_dir)
    df_dict = format_datetime(df_dict)
    df_dict = optimize_datatypes(df_dict)
    df_dict = handle_missing_values(df_dict)
    
    print("Data Preprocessing Completed!")
    return df_dict

In [26]:
# Tự dò đường dẫn dữ liệu để chạy ổn định ở cả notebook folder và workspace root
data_dir_candidates = [Path('../data/raw'), Path('data/raw'), Path('./data/raw')]
DATA_DIR = next((p.resolve() for p in data_dir_candidates if p.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError('Khong tim thay thu muc data/raw. Hay kiem tra cau truc du an.')

print(f'Using DATA_DIR: {DATA_DIR}')

# 1. Thực thi luồng xử lý và lưu toàn bộ bảng vào dictionary
clean_data = preprocess_data(DATA_DIR)

# 2. Trích xuất các bảng cụ thể để sử dụng
df_customers = clean_data.get('customers')
df_orders = clean_data.get('orders')
df_orderitems = clean_data.get('orderitems')
df_inventory = clean_data.get('inventory')

print(f'Tong so bang da nap: {len(clean_data)}')

# 3. Kiểm thử nhanh
if df_customers is not None:
    print('--- Thong tin bang customers ---')
    print(df_customers.info())
    
    print('\n--- 5 dong dau tien ---')
    print(df_customers.head())
    
    if 'gender' in df_customers.columns:
        print('\n--- Kiem tra gia tri khuyet thieu da duoc dien thanh Unknown ---')
        print(df_customers['gender'].value_counts())

if df_orderitems is not None:
    print('\n--- 5 dong dau bang orderitems ---')
    preview_cols = [col for col in ['orderid', 'productid', 'promoid', 'promoid2'] if col in df_orderitems.columns]
    print(df_orderitems[preview_cols].head())

if df_inventory is not None:
    print('\n--- 5 dong dau bang inventory ---')
    inventory_preview_cols = [
        col for col in ['snapshotdate', 'productid', 'stockonhand', 'unitsreceived', 'unitssold']
        if col in df_inventory.columns
    ]
    print(df_inventory[inventory_preview_cols].head())

# 4. Xuất bộ CSV đã chuẩn hóa theo format đề sang data/processed
output_dir_candidates = [Path('../data/processed'), Path('data/processed'), Path('./data/processed')]
OUTPUT_DIR = next((p.resolve() for p in output_dir_candidates if p.exists()), output_dir_candidates[0].resolve())
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for table_name, df in clean_data.items():
    output_path = OUTPUT_DIR / f'{table_name}.csv'
    df.to_csv(output_path, index=False)

print(f'\nDa xuat {len(clean_data)} file CSV da chuan hoa vao: {OUTPUT_DIR}')

Using DATA_DIR: D:\datathon-hkbaleycb4\data\raw
Loaded customers from customers.csv: (121930, 7)
Loaded geography from geography.csv: (39948, 4)
Loaded inventory from inventory.csv: (60247, 17)
Loaded orderitems from order_items.csv: (714669, 7)
Loaded orders from orders.csv: (646945, 8)
Loaded payments from payments.csv: (646945, 4)
Loaded products from products.csv: (2412, 8)
Loaded promotions from promotions.csv: (50, 10)
Loaded returns from returns.csv: (39939, 7)
Loaded reviews from reviews.csv: (113551, 7)
Loaded sales from sales.csv: (3833, 3)
Loaded samplesubmission from sample_submission.csv: (548, 3)
Loaded shipments from shipments.csv: (566067, 4)
Loaded webtraffic from web_traffic.csv: (3652, 7)
Data Preprocessing Completed!
Tong so bang da nap: 14
--- Thong tin bang customers ---
<class 'pandas.DataFrame'>
RangeIndex: 121930 entries, 0 to 121929
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              -------------- 